# HeritageAR YOLOv8 Training - Google Colab
## Function 1: Sigiriya Sub-Landmark Detection

This notebook is the Colab-only training path for the HeritageAR backend.

Use it to:
1. Mount Google Drive
2. Clone the repository
3. Install backend dependencies
4. Bring in the Roboflow YOLOv8 dataset
5. Train and evaluate the YOLOv8 model
6. Convert the best checkpoint to TFLite
7. Download the final model for Flutter

Do not use this notebook for local environment setup.


## Step 1: Mount Google Drive and Clone the Repository
This keeps the trained model and exported files in Drive while Colab handles training.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive so you can save checkpoints and downloaded models
# to persistent storage.
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")


In [ ]:
# Clone the project repository if it is not already present.
repo_url = "YOUR_REPOSITORY_URL_HERE"
repo_dir = Path("/content/heritage_ar")

if not repo_dir.exists():
    !git clone {repo_url} {repo_dir}
else:
    print(f"✓ Repository already exists at {repo_dir}")

os.chdir(repo_dir)
print("Current directory:", os.getcwd())


## Step 2: Install Backend Dependencies
Install the Python packages required for training, evaluation, and export.


In [ ]:
# Install project dependencies from the backend requirements file.
!pip install -q -r backend/requirements.txt
!pip install -q ultralytics
print("✓ Backend dependencies installed successfully!")


## Step 3: Add the Roboflow YOLOv8 Dataset
Export the annotated dataset from Roboflow as YOLOv8, then place the extracted folder in Colab or Drive.


In [ ]:
import os
from pathlib import Path

# The dataset should be extracted to /content/sigiriya_dataset
# with this structure:
# /content/sigiriya_dataset/images/{train,val,test}
# /content/sigiriya_dataset/labels/{train,val,test}
dataset_path = Path("/content/sigiriya_dataset")
dataset_path.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", dataset_path)
print("Expected class names:")
print("0: sigiriya_entrance")
print("1: sigiriya_lion_rock")
print("2: sigiriya_mirror_wall")
print("3: sigiriya_lion_staircase")
print("4: sigiriya_throne")


## Step 4: Upload Roboflow Dataset
⚠️ **IMPORTANT:**
1. Download your annotated dataset from Roboflow as "YOLOv8" format (`.zip`)
2. Run the cell below
3. Upload the `.zip` file when prompted
4. Wait for extraction to complete

In [ ]:
from google.colab import files
import zipfile

print("Uploading Roboflow dataset ZIP file...")
print("Click 'Choose Files' and select your exported YOLOv8 dataset")

uploaded = files.upload()

# Get the uploaded zip file
zip_file = list(uploaded.keys())[0]
print(f"✓ Uploaded: {zip_file}")

# Extract to sigiriya_dataset
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content')

print("✓ Dataset extracted!")

# List contents
!ls -la /content/sigiriya_dataset/

## Step 5: Create or Update data.yaml
This tells YOLOv8 where to find the training data.

In [ ]:
# Create data.yaml for Colab
data_yaml_content = """# YOLOv8 dataset config for Sigiriya sub-landmark detection
# Google Colab path
path: /content/sigiriya_dataset
train: images/train
val: images/val
test: images/test

nc: 5  # number of classes

names:
  0: sigiriya_entrance
  1: sigiriya_lion_rock
  2: sigiriya_mirror_wall
  3: sigiriya_lion_staircase
  4: sigiriya_throne
"""

with open('/content/sigiriya_dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("✓ data.yaml created!")
print("\nContents:")
!cat /content/sigiriya_dataset/data.yaml


## Step 6: Verify Dataset Structure

In [ ]:
# Verify dataset structure
import os

dataset_path = '/content/sigiriya_dataset'

print("Dataset Structure:")
print("="*50)

# Check images
for split in ['train', 'val', 'test']:
    img_path = f'{dataset_path}/images/{split}'
    label_path = f'{dataset_path}/labels/{split}'
    
    if os.path.exists(img_path):
        img_count = len(os.listdir(img_path))
        label_count = len(os.listdir(label_path)) if os.path.exists(label_path) else 0
        print(f"✓ {split.upper()}: {img_count} images, {label_count} labels")
    else:
        print(f"❌ {split.upper()}: Missing folder")

print("\n✓ Dataset verification complete!")

## Step 7: Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Load YOLOv8n (nano - best for mobile)
model = YOLO('yolov8n.pt')

print("Starting YOLOv8 training...")
print("This may take 30-60 minutes on Colab GPU")
print("="*50)

# Train the model
results = model.train(
    data='/content/sigiriya_dataset/data.yaml',
    epochs=100,                    # Number of training epochs
    imgsz=640,                     # Input image size
    batch=16,                      # Batch size (increase if GPU allows)
    patience=20,                   # Early stopping patience
    device=0,                      # GPU device
    project='/content/runs/detect',
    name='sigiriya_v1',
    save=True,
    verbose=True
)

print("\n✓ Training complete!")
print(f"Best model saved to: /content/runs/detect/sigiriya_v1/weights/best.pt")

## Step 4: Train the YOLOv8 Model
Run the repository training script against the Colab dataset.


In [ ]:
# Train the model using the repository script.
!python backend/training/train_sigiriya_yolov8.py


## Step 6: Optional Export to ONNX
Use the repository conversion script if you want an ONNX checkpoint for experimentation.


In [ ]:
# Export the best checkpoint to ONNX using the repository script.
!python backend/conversion/pytorch_to_onnx.py --input_model runs/detect/sigiriya_v1/weights/best.pt --output_model backend/output/sigiriya_best.onnx


## Step 7: Optional Export to TFLite
Use the repository conversion script to create the Flutter-ready model.


In [ ]:
# Export the best checkpoint to TFLite using the repository script.
!python backend/conversion/onnx_to_tflite.py --input_model runs/detect/sigiriya_v1/weights/best.pt --output_model backend/output/sigiriya_best.tflite


## Step 11: Convert ONNX to TensorFlow Lite

In [ ]:
print("Converting ONNX to TensorFlow Lite...")
print("This may take 5-10 minutes...\n")

import subprocess

# Use tf2onnx and then convert to TFLite
onnx_model_path = '/content/sigiriya_best.onnx'
pb_model_path = '/content/sigiriya_model.pb'
tflite_model_path = '/content/sigiriya_best.tflite'

# Method: Use ONNX to TFLite conversion
!pip install -q tf2onnx onnx-tf

import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

# Load ONNX model
onnx_model = onnx.load(onnx_model_path)
onnx.checker.check_model(onnx_model)

# Convert to TensorFlow
tf_rep = prepare(onnx_model)
tf_rep.export_graph(pb_model_path)

print("✓ ONNX to TensorFlow conversion complete!")

## Step 12: Quantize Model (Optional but Recommended)

In [ ]:
# Simple approach: Use ONNX to TFLite directly
print("Converting to TensorFlow Lite format...\n")

# Use YOLOv8's built-in TFLite export
model_path = '/content/runs/detect/sigiriya_v1/weights/best.pt'
model = YOLO(model_path)

try:
    # Export to TFLite (quantized)
    tflite_model = model.export(
        format='tflite',
        imgsz=640,
        half=False,  # Set to True for FP16 quantization
        int8=False   # Set to True for INT8 quantization
    )
    print(f"✓ TFLite model exported successfully!")
    print(f"Model location: {tflite_model}")
except Exception as e:
    print(f"Note: TFLite export had an issue: {e}")
    print("Continue to next step for manual conversion")

## Step 13: Find and Prepare Final Model

In [ ]:
import os

print("Looking for TFLite model...\n")

# Search for TFLite models
tflite_files = []
for root, dirs, files in os.walk('/content/runs'):
    for file in files:
        if file.endswith('.tflite'):
            tflite_files.append(os.path.join(root, file))

if tflite_files:
    print(f"Found {len(tflite_files)} TFLite model(s):")
    for f in tflite_files:
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  ✓ {f} ({size_mb:.2f} MB)")
else:
    print("No TFLite models found yet. Using PyTorch model instead.")
    pt_file = '/content/runs/detect/sigiriya_v1/weights/best.pt'
    if os.path.exists(pt_file):
        size_mb = os.path.getsize(pt_file) / (1024*1024)
        print(f"\nAvailable: {pt_file} ({size_mb:.2f} MB)")

## Step 14: Download Models

In [ ]:
from google.colab import files
import os

print("Preparing models for download...\n")

# Download PyTorch model (always available)
pt_model = '/content/runs/detect/sigiriya_v1/weights/best.pt'
if os.path.exists(pt_model):
    print(f"Downloading: {pt_model}")
    files.download(pt_model)
    print("✓ PyTorch model downloaded!")

# Download TFLite if available
tflite_models = []
for root, dirs, files_list in os.walk('/content/runs'):
    for file in files_list:
        if file.endswith('.tflite'):
            full_path = os.path.join(root, file)
            print(f"\nDownloading: {full_path}")
            files.download(full_path)
            print("✓ TFLite model downloaded!")

print("\n" + "="*50)
print("All models downloaded to your computer!")
print("="*50)

## Summary

Use this notebook to train in Colab, evaluate the checkpoint, and export the final model from the repository scripts.

Recommended output for Flutter:
- `backend/output/sigiriya_best.tflite`
- or `backend/output/sigiriya_best_quantized.tflite`

After download, copy the final `.tflite` file into `frontend/assets/models/landmark_model.tflite`.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# Prefer the quantized model if it exists, otherwise fall back to the plain TFLite export.
quantized_model = Path("backend/output/sigiriya_best_quantized.tflite")
plain_model = Path("backend/output/sigiriya_best.tflite")
model_to_download = quantized_model if quantized_model.exists() else plain_model

if model_to_download.exists():
    print(f"✓ Downloading {model_to_download}")
    files.download(str(model_to_download))

    drive_output = Path("/content/drive/MyDrive/HeritageAR_Models")
    drive_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(model_to_download, drive_output / model_to_download.name)
    print(f"✓ Copied to Drive: {drive_output / model_to_download.name}")
else:
    print("No exported TFLite model found yet. Run the export cells first.")


## Summary

✅ **Training Complete!**

### Downloaded Models:
- **best.pt** - PyTorch model (for future training/conversion)
- **best.tflite** (if generated) - Mobile-ready model

### Next Steps:
1. Download the `.tflite` model
2. Copy to `frontend/assets/models/landmark_model.tflite`
3. Run Flutter app on Android device with ARCore support

### Monitor Results:
- Check `runs/detect/sigiriya_v1/results.csv` for detailed metrics
- Review training plots for convergence

---
**Project:** HeritageAR - Function 1  
**Date:** April 2026  
**Status:** ✅ Complete